# Experimenting with the LEDnetWF020027A5A9B7 sunset lamp

Yay!

In [2]:
# Set this to the MAC address of your LED. 
address = '08:65:f0:a5:a9:b7'

# Understanding colors

In [3]:
# Here are our colors commands:
red =    bytearray.fromhex('0001 8000 000D 0E0B 3BA1 B464 6400 0000 0014 0000 6C')
purple = bytearray.fromhex('0002 8000 000D 0E0B 3BA1 9664 6400 0000 0014 0000 4E')
blue =   bytearray.fromhex('0003 8000 000D 0E0B 3BA1 7864 6400 0000 0014 0000 30')
cyan =   bytearray.fromhex('0004 8000 000D 0E0B 3BA1 5A64 6400 0000 0014 0000 12')
green =  bytearray.fromhex('0005 8000 000D 0E0B 3BA1 3C64 6400 0000 0014 0000 F4')
yellow = bytearray.fromhex('0006 8000 000D 0E0B 3BA1 1E64 6400 0000 0014 0000 D6')

# Reddish?
a5 = bytearray.fromhex('0007 8000 000D 0E0B 3BA1 0064 6400 0000 0014 0000 B8')

all_colors = [red, purple, blue, cyan, green, yellow, a5]

In [4]:
# We already know that the first 2 bytes are a counter.
# The last byte is a checksum from [`3BA1`, end).

# So the only "color" bytes are the 3rd to 10th bytes.

In [5]:
# Set up Bleak etc
import asyncio
from bleak import BleakClient, BleakScanner

COLOR_UUID = '0000ff01-0000-1000-8000-00805f9b34fb'

counter = 1

def set_counter(packet: bytearray):
    global counter
    packet[1] = counter & 0xff
    packet[0] = (counter >> 8) & 0xff
    counter += 1
    return packet

def set_checksum(packet: bytearray):
    # The checksum is the last byte of the packet.
    # It is the sum of bytes 8 until end (excluding the checksum byte) modulo 256.
    original_checksum = packet[-1]
    packet[-1] = 0  # Reset checksum byte
    checksum = sum(packet[8:]) % 256
    packet[-1] = checksum
    # if original_checksum != checksum:
    #     print(f'Checksum mismatch: original {original_checksum:02x}, calculated {checksum:02x}')
    # else:
    #     print(f'Checksum matches: {checksum:02x}')
    return packet

In [8]:
device = await BleakScanner.find_device_by_address(address)

async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for color in all_colors:
        packet = set_counter(color.copy())
        packet = set_checksum(packet)
        print(f'Sending packet: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.5)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending packet: 00018000000d0e0b3ba1b46464000000001400006c
Sending packet: 00028000000d0e0b3ba1966464000000001400004e
Sending packet: 00038000000d0e0b3ba17864640000000014000030
Sending packet: 00048000000d0e0b3ba15a64640000000014000012
Sending packet: 00058000000d0e0b3ba13c646400000000140000f4
Sending packet: 00068000000d0e0b3ba11e646400000000140000d6
Sending packet: 00078000000d0e0b3ba100646400000000140000b8


In [9]:

async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    packet = set_counter(blue.copy())
    packet = set_checksum(packet)
    print(f'Sending packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending packet: 00088000000d0e0b3ba17864640000000014000030


In [32]:
# e.g. generate_color_packet(bytearray.fromhex('3c64'))
def generate_color_packet(color_byte: bytearray) -> bytearray:
    # start with red
    packet = red.copy()
    # replace the color bytes with the provided color byte
    bytes_to_replace = len(color_byte)
    packet[10:10+bytes_to_replace] = color_byte[:bytes_to_replace]
    # set the counter and checksum
    packet = set_counter(packet)
    packet = set_checksum(packet)
    return packet


In [ ]:

async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for bytes in range(0, 256, 10):
        color_byte = bytearray.fromhex(f'{bytes:02x} 64')  # e.g. '3c64' for 60
        packet = generate_color_packet(color_byte)
        print(f'Sending packet for color byte {bytes:02x}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.5)


## CYCLES THROUGH COLORS! YAY! Deriving from the previous investigations, the range is 0-b4 (0-180), which is 0-360 divided by 2. It loops after that.

In [29]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for bytes in range(0, 100, 5):
        # This time, hold the first byte constant and vary the second byte.
        color_byte = bytearray.fromhex(f'78 {bytes:02x}')
        packet = generate_color_packet(color_byte)
        print(f'Sending packet for color byte {bytes:02x}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.5)

# Seems to vary the whiteness; 100 (0x64) is blue, 0 is white. Then it loops?

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending packet for color byte 00: 00ae8000000d0e0b3ba178006400000000140000cc
Sending packet for color byte 05: 00af8000000d0e0b3ba178056400000000140000d1
Sending packet for color byte 0a: 00b08000000d0e0b3ba1780a6400000000140000d6
Sending packet for color byte 0f: 00b18000000d0e0b3ba1780f6400000000140000db
Sending packet for color byte 14: 00b28000000d0e0b3ba178146400000000140000e0
Sending packet for color byte 19: 00b38000000d0e0b3ba178196400000000140000e5
Sending packet for color byte 1e: 00b48000000d0e0b3ba1781e6400000000140000ea
Sending packet for color byte 23: 00b58000000d0e0b3ba178236400000000140000ef
Sending packet for color byte 28: 00b68000000d0e0b3ba178286400000000140000f4
Sending packet for color byte 2d: 00b78000000d0e0b3ba1782d6400000000140000f9
Sending packet for color byte 32: 00b88000000d0e0b3ba178326400000000140000fe
Sending packet for color byte 37: 00b98000000d0e0b3ba17837640000000014000003
Sending packet for co

In [34]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for bytes in range(0, 256, 10):
        # This time, hold the first 2 bytes constant and vary the third byte.
        color_byte = bytearray.fromhex(f'78 64 {bytes:02x}')
        packet = generate_color_packet(color_byte)
        print(f'Sending packet for color byte {bytes:02x}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.5)

# Seems to vary the brightness; 100 (0x64) is full, 0 is off. Then it loops?

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending packet for color byte 00: 01108000000d0e0b3ba178640000000000140000cc
Sending packet for color byte 0a: 01118000000d0e0b3ba178640a00000000140000d6
Sending packet for color byte 14: 01128000000d0e0b3ba178641400000000140000e0
Sending packet for color byte 1e: 01138000000d0e0b3ba178641e00000000140000ea
Sending packet for color byte 28: 01148000000d0e0b3ba178642800000000140000f4
Sending packet for color byte 32: 01158000000d0e0b3ba178643200000000140000fe
Sending packet for color byte 3c: 01168000000d0e0b3ba178643c0000000014000008
Sending packet for color byte 46: 01178000000d0e0b3ba17864460000000014000012
Sending packet for color byte 50: 01188000000d0e0b3ba1786450000000001400001c
Sending packet for color byte 5a: 01198000000d0e0b3ba178645a0000000014000026
Sending packet for color byte 64: 011a8000000d0e0b3ba17864640000000014000030
Sending packet for color byte 6e: 011b8000000d0e0b3ba178646e000000001400003a
Sending packet for co

In [37]:
ON_PACKET              = bytearray.fromhex("00 04 80 00 00 0d 0e 0b 3b 23 00 00 00 00 00 00 00 32 00 00 90")
OFF_PACKET             = bytearray.fromhex("00 5b 80 00 00 0d 0e 0b 3b 24 00 00 00 00 00 00 00 32 00 00 91")

async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    # Turn on the light
    packet = set_counter(ON_PACKET.copy())
    packet = set_checksum(packet)
    print(f'Sending ON packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

    # Turn off the light
    packet = set_counter(OFF_PACKET.copy())
    packet = set_checksum(packet)
    print(f'Sending OFF packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

    packet = set_counter(ON_PACKET.copy())
    packet = set_checksum(packet)
    print(f'Sending ON packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

# Yup, these work too (stolen from lefwf_controller.py).

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending ON packet: 012e8000000d0e0b3b230000000000000032000090
Sending OFF packet: 012f8000000d0e0b3b240000000000000032000091
Sending ON packet: 01308000000d0e0b3b230000000000000032000090


In [ ]:
# I wonder what that 14 is for.

# B464 6400 0000 0014
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    # for value in range(0, 256, 10):
    # value = 0xff0
    packet = generate_color_packet(bytearray.fromhex(f'B464 6400 0000 0000'))
    print(f'Sending packet for on: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(3)

    # value = 0x100 # About 3 seconds
    # value = 0x500 # about 14 seconds
    # value = 256 # about 2.5 seconds
    # value = 512 # about 5 seconds
    # value = 700 # about 7 seconds
    value = 1000 # about 10 seconds
    # 0064 6400 0000 3314
    packet = generate_color_packet(bytearray.fromhex(f'B464 0000 0000 {value:04x}'))
    print(f'Sending packet for off (t={value}): {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    # await asyncio.sleep(3)

# Duration, in 10ms increments, which matches other standards (e.g. Zigbee uses 10ms increments).

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending packet for on: 035c8000000d0e0b3ba1b464640000000000000058
Sending packet for off (t=1000): 035d8000000d0e0b3ba1b4640000000003e80000df


In [125]:
def generate_hsv_packet(hue: int, saturation: int, value: int, duration_10ms: int = 14) -> bytearray:
    if not (0 <= hue <= 360):
        raise ValueError("Hue must be between 0 and 360")
    if not (0 <= saturation <= 100):
        raise ValueError("Saturation must be between 0 and 100")
    if not (0 <= value <= 100):
        raise ValueError("Value must be between 0 and 100")
    if not (0 <= duration_10ms <= 0xffff):
        raise ValueError("Duration must be between 0 and 65535 (0-0xffff)")

    half_hue = int(hue / 2)  # Convert to the range used by the device (0-180)

    return generate_color_packet(
        bytearray.fromhex(f'{half_hue:02x} {saturation:02x} {value:02x}00 0000 {duration_10ms:04x}'))

In [ ]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for hue in range(0, 360, 10):
        packet = generate_hsv_packet(hue, 100, 100)
        print(f'Sending HSV packet for hue {hue}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet, response = True)
        await asyncio.sleep(0.25)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending HSV packet for hue 0: 020b8000000d0e0b3ba100646400000000140000b8
Sending HSV packet for hue 10: 020c8000000d0e0b3ba105646400000000140000bd
Sending HSV packet for hue 20: 020d8000000d0e0b3ba10a646400000000140000c2
Sending HSV packet for hue 30: 020e8000000d0e0b3ba10f646400000000140000c7
Sending HSV packet for hue 40: 020f8000000d0e0b3ba114646400000000140000cc
Sending HSV packet for hue 50: 02108000000d0e0b3ba119646400000000140000d1
Sending HSV packet for hue 60: 02118000000d0e0b3ba11e646400000000140000d6
Sending HSV packet for hue 70: 02128000000d0e0b3ba123646400000000140000db
Sending HSV packet for hue 80: 02138000000d0e0b3ba128646400000000140000e0
Sending HSV packet for hue 90: 02148000000d0e0b3ba12d646400000000140000e5
Sending HSV packet for hue 100: 02158000000d0e0b3ba132646400000000140000ea
Sending HSV packet for hue 110: 02168000000d0e0b3ba137646400000000140000ef
Sending HSV packet for hue 120: 02178000000d0e0b3ba13c64

In [52]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for saturation in range(0, 101, 10):
        packet = generate_hsv_packet(0, saturation, 100)
        print(f'Sending HSV packet for saturation {saturation:02d}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.25)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending HSV packet for saturation 00: 023a8000000d0e0b3ba10000640000000014000054
Sending HSV packet for saturation 10: 023b8000000d0e0b3ba1000a64000000001400005e
Sending HSV packet for saturation 20: 023c8000000d0e0b3ba10014640000000014000068
Sending HSV packet for saturation 30: 023d8000000d0e0b3ba1001e640000000014000072
Sending HSV packet for saturation 40: 023e8000000d0e0b3ba1002864000000001400007c
Sending HSV packet for saturation 50: 023f8000000d0e0b3ba10032640000000014000086
Sending HSV packet for saturation 60: 02408000000d0e0b3ba1003c640000000014000090
Sending HSV packet for saturation 70: 02418000000d0e0b3ba1004664000000001400009a
Sending HSV packet for saturation 80: 02428000000d0e0b3ba100506400000000140000a4
Sending HSV packet for saturation 90: 02438000000d0e0b3ba1005a6400000000140000ae
Sending HSV packet for saturation 100: 02448000000d0e0b3ba100646400000000140000b8


In [ ]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for value in range(0, 101, 10):
        packet = generate_hsv_packet(0, 100, value)
        print(f'Sending HSV packet for value {value:02d}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.25)


Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending HSV packet for value 00: 02508000000d0e0b3ba10064000000000014000054
Sending HSV packet for value 10: 02518000000d0e0b3ba100640a000000001400005e
Sending HSV packet for value 20: 02528000000d0e0b3ba10064140000000014000068
Sending HSV packet for value 30: 02538000000d0e0b3ba100641e0000000014000072
Sending HSV packet for value 40: 02548000000d0e0b3ba1006428000000001400007c
Sending HSV packet for value 50: 02558000000d0e0b3ba10064320000000014000086
Sending HSV packet for value 60: 02568000000d0e0b3ba100643c0000000014000090
Sending HSV packet for value 70: 02578000000d0e0b3ba1006446000000001400009a
Sending HSV packet for value 80: 02588000000d0e0b3ba100645000000000140000a4
Sending HSV packet for value 90: 02598000000d0e0b3ba100645a00000000140000ae
Sending HSV packet for value 100: 025a8000000d0e0b3ba100646400000000140000b8


In [128]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    # for duration_ms in range(0, 65536, 1000): # 1000ms = 1 second
    for duration_ms in range(0, 10000, 1000): # 1000ms = 1 second
        packet = generate_hsv_packet(0, 100, 100, duration_10ms=0)
        print(f'Turning on                          : {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.25)

        packet = generate_hsv_packet(0, 100, 0, duration_10ms=duration_ms // 10)
        print(f'Sending HSV packet for duration {duration_ms: 4}ms: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(duration_ms / 1000)  # Sleep for the duration in seconds



Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Turning on                          : 03ba8000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration    0ms: 03bb8000000d0e0b3ba10064000000000000000040
Turning on                          : 03bc8000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration  1000ms: 03bd8000000d0e0b3ba100640000000000640000a4
Turning on                          : 03be8000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration  2000ms: 03bf8000000d0e0b3ba100640000000000c8000008
Turning on                          : 03c08000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration  3000ms: 03c18000000d0e0b3ba1006400000000012c00006d
Turning on                          : 03c28000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration  4000ms: 03c38000000d0e0b3ba100640000000001900000d1
Turning on                          : 03c48000000d0e0b3ba100646400000000000000a4
Sending HSV packet for duration  5000ms:

## White vs colors

The bulb is RGBWW, so you can control the white separately.

In [17]:
ww_on =    bytearray.fromhex('005C 8000 000D 0E0B 3BB1 0000 0000 6400 001E 0000 6E')
gryellow = bytearray.fromhex('005D 8000 000D 0E0B 3BA1 3C64 6400 0000 0014 0000 F4')
ww_off =   bytearray.fromhex('005E 8000 000D 0E0B 3BB1 0000 0000 0000 001E 0000 6F')

async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    # Turn to green
    packet = set_counter(gryellow.copy())
    packet = set_checksum(packet)
    print(f'Sending gryellow packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

    # Turn on the warm white light
    packet = set_counter(ww_on.copy())
    packet = set_checksum(packet)
    print(f'Sending WW ON packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

    # Turn off the warm white light
    packet = set_counter(ww_off.copy())
    packet = set_checksum(packet)
    print(f'Sending WW ON packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

    # Turn it back on
    packet = set_counter(ww_on.copy())
    packet = set_checksum(packet)
    print(f'Sending WW ON packet: {packet.hex()}')
    await client.write_gatt_char(COLOR_UUID, packet)
    await asyncio.sleep(0.5)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending gryellow packet: 00188000000d0e0b3ba13c646400000000140000f4
Sending WW ON packet: 00198000000d0e0b3bb1000000006400001e00006e
Sending WW ON packet: 001a8000000d0e0b3bb1000000000000001e00000a
Sending WW ON packet: 001b8000000d0e0b3bb1000000006400001e00006e


In [20]:
def generate_white_packet(percentage: int) -> bytearray:
    if not (0 <= percentage <= 100):
        raise ValueError("Percentage must be between 0 and 100")

    base = bytearray.fromhex('005C 8000 000D 0E0B 3BB1 0000 0000 6400 001E 0000 6E')
    # Replace the 64 with the percentage value
    packet = base.copy()
    packet[14] = percentage  # Set the percentage in the packet
    packet = set_counter(packet)
    packet = set_checksum(packet)
    return packet

In [ ]:
async with BleakClient(device) as client:
    print(f'Connected to {device.name} at {device.address}')

    for percentage in range(0, 101, 10):
        packet = generate_white_packet(percentage)
        print(f'Sending white packet for percentage {percentage:02d}: {packet.hex()}')
        await client.write_gatt_char(COLOR_UUID, packet)
        await asyncio.sleep(0.25)

Connected to LEDnetWF020027A5A9B7 at 08:65:F0:A5:A9:B7
Sending white packet for percentage 00: 00328000000d0e0b3bb1000000000000001e00000a
Sending white packet for percentage 10: 00338000000d0e0b3bb1000000000a00001e000014
Sending white packet for percentage 20: 00348000000d0e0b3bb1000000001400001e00001e
Sending white packet for percentage 30: 00358000000d0e0b3bb1000000001e00001e000028
Sending white packet for percentage 40: 00368000000d0e0b3bb1000000002800001e000032
Sending white packet for percentage 50: 00378000000d0e0b3bb1000000003200001e00003c
Sending white packet for percentage 60: 00388000000d0e0b3bb1000000003c00001e000046
Sending white packet for percentage 70: 00398000000d0e0b3bb1000000004600001e000050
Sending white packet for percentage 80: 003a8000000d0e0b3bb1000000005000001e00005a
